# Checkpoint 1 Guidance: Preparing Real Data

Simulated data arrives clean. Real data has missing values, confusing codes, and categories stored as text. This notebook walks through one real data set from start to finish. The story in one line: **before any model, look at the data; then put every step that learns from data inside one pipeline, so cross-validation stays honest.**

This notebook is a reference for Kaggle Checkpoint 1: read it on your own, and come back to it while you work on `checkpoint1.ipynb`, the starter notebook in this folder. Unlike the lecture notebooks, almost all of its code is worth reading, because it's code you'll write yourself.

### By the end, you'll be able to

1. Read a raw CSV file with `pd.read_csv` and catch two common loading problems: missing-value codes that pandas doesn't recognize, and True/False columns with gaps.
2. Look at a new data set and find its missing values, its strange or unexpected values, and how balanced its classes are.
3. Choose between dropping and filling in (imputing) missing values, and explain why an imputer must be fit inside cross-validation.
4. Decide what to do about an outlier by first asking what the value means.
5. Turn categorical features into numbers with one-hot encoding, and explain why numbering the categories 0, 1, 2 is a problem for a linear model.
6. Build one pipeline that imputes, scales, encodes, and fits a model, and cross-validate it.

In [ ]:
import numpy as np
import pandas as pd

**The data.** The Adult census data: about 49,000 adults from the 1994 US census, with the task of predicting whether a person's income is above \$50,000 a year. Like your Kaggle project, it's a classification task on a table of mixed numeric and text columns.

**Code worth reading.** `fetch_openml` downloads the data from OpenML the first time (you need an internet connection), then loads it from a saved copy afterward. `.frame` is the data as a pandas DataFrame.

We keep 10 of the columns. In the Kaggle assignment, keep every column unless you can say why it can't help in its current form. Before you drop a column that looks like an ID or a name, check whether it packs in a piece of potentially useful information you could pull out into a separate column. For example, in the original Titanic dataset, names contained *honorifics* like "Mr", "Ms" and others. These help impute missing values, when gender and/or age were unknown.

In [ ]:
from sklearn.datasets import fetch_openml

adult = fetch_openml('adult', version=2, as_frame=True).frame # download the Adult dataset from OpenML

# You'll actually need to examine columns to decide which features to use initially. We skip that step here.
columns = ['age', 'workclass', 'education-num', 'marital-status', 'occupation',
           'relationship', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
X = adult[columns]
y = (adult['class'] == '>50K').astype(int)      # 1 = income above $50K, 0 = not
X.head()

<a id="section-0"></a>

## 0 · Reading a raw file

`fetch_openml` loaded data that was already cleaned up. OpenML:

- turned the original file's missing-value codes into proper missing values (`NaN`) 
- gave each column a sensible type. 

Your own data, like the Kaggle project's `train.csv`, arrives as a raw CSV file that you read with `pd.read_csv`, and that's where two of the most common data problems start.

**Below, we create those problems on purpose, so you can see how to fix them.** `write_raw_adult_csv` writes `X` back out as the kind of data file you'll start with in Checkpoint 1:

- **Missing values that `.isna()` can't detect.** In the text columns `workclass`, `occupation`, and `native-country`, the missing values are written as a non-standard character, instead of being left blank, so pandas reads them as ordinary values. We'll work through how to discover and fix this problem below.
- **An extra True/False column with gaps.** `born_in_us` is blank wherever `native-country` is missing. It's only there to carry this second problem, and we drop it at the end of this section.

**Code worth reading.** `pd.read_csv` reads a CSV file into a DataFrame, and `.isna().sum()` counts the missing values in each column.

In [ ]:
from checkpoint_1_helpers import write_raw_adult_csv # You don't need to read this function.

raw_path = write_raw_adult_csv(X)     # the path of the file it wrote
raw = pd.read_csv(raw_path)
raw.isna().sum()

**Ask:** Look at the counts for the text columns, like `workclass` and `occupation`. Is anything suspicious?

<details>
<summary>Answer</summary>

Zero missing values in a survey of about 49,000 people is too good to be true: real survey data almost always has gaps. `pd.read_csv` only treats blank fields and a few standard codes (like `NA`, `NaN`, and `null`) as missing. If a file marks missing values some other way, pandas reads them as ordinary values, and `.isna()` can't see them.

</details>

### How to find out how a file marks missing values

1. **Read the documentation.** Most data sets come with a description of each column (for the Kaggle project, it's the competition's **Data** tab), and it often says how missing values are written.
2. **Look at the values themselves**, because documentation is often incomplete. A stand-in for "missing," called a **sentinel value**, usually stands out: a symbol, a very short value, or a word like `unknown`, in a column where every other value is a real answer. In numeric columns, look for impossible minimums or maximums like `-1` or `999` in `.describe()` (Section 1). And a column of numbers that initially loads with type `object` (text) almost always has a text code mixed in.


Let's start by identifying which columns loaded as numbers and which as objects:

In [ ]:
# Display raw's data types
raw.dtypes


**Code worth reading.** `select_dtypes(include='object')` picks out the object (text) columns. For each one, the loop prints every distinct value, sorted.

In [ ]:
for column in raw.select_dtypes(include='object').columns:
    values = sorted(raw[column].dropna().astype(str).unique())
    print(f'{column} ({len(values)} values): {", ".join(values)}\n')

**Ask:** Which value in this output stands for "missing" rather than a real answer?

<details>
<summary>Answer</summary>

`?`. It shows up in `workclass`, `occupation`, and `native-country`, and nobody's occupation is "?". (Sorting helps: `?` sorts before the letters, so it comes first in each list.) `value_counts` shows how many rows use it.

</details>

In [ ]:
raw['workclass'].value_counts().head()

Notice that 2799 values in the `workclass` column are written as `?`, so those are missing.

### Problem 1: a missing-value code pandas doesn't know

If you don't catch this, a model treats `?` as one more job type, and every count of missing values you report is wrong. The fix is to tell `pd.read_csv` about the code.

**Code worth reading.** `na_values='?'` adds `?` to the codes that `pd.read_csv` treats as missing. In other data, look for `''`, `-1`, `999`, `unknown`, or `N/A`, and pass whichever codes you find (as a list, if there are several).

A related tool: if a numeric column loaded as `object` because of a text code, `pd.to_numeric(raw[column], errors='coerce')` converts it to numbers and **coerces** anything that isn't a number into `NaN`. Before you coerce, list exactly the values that would become `NaN`: the ones that aren't missing now but would be afterward.

```python
numbers = pd.to_numeric(raw[column], errors='coerce')
raw.loc[numbers.isna() & raw[column].notna(), column].value_counts()
```

<details>
<summary>How the check works</summary>

`.isna()` is `True` wherever a value is missing, and `.notna()` is its opposite: `True` wherever a value is present. Each gives one `True` or `False` per row. The `&` combines two of them row by row and keeps `True` only where both are `True`. (Use `&`, not the word `and`, which doesn't work on whole columns.) So `numbers.isna() & raw[column].notna()` means "missing after coercing, but not missing before," which is exactly the set of values that coercing would turn into `NaN`. `raw.loc[..., column]` keeps just those rows of the column, and `.value_counts()` counts each distinct value among them.

For example, take a column holding `'12'`, `'?'`, `'1,200'`, a blank, and `'7'`. Coercing turns `'?'`, `'1,200'`, and the blank into `NaN`. The blank was already missing, so only `'?'` and `'1,200'` pass the check, and those are the values to look at.

</details>

If the list holds only sentinels like `?` or `unknown`, coercing is safe. If it holds real values in an odd format, like `1,200` or `$45`, coercing would quietly erase real data, so fix those first (for example, by removing the commas). We skip this below and just do the value coercion (`na_values='?'`) because we already know how many missing values to expect. In Kaggle, do something like the check explained above.

In [ ]:
raw = pd.read_csv(raw_path, na_values='?')
raw.isna().sum()

The missing values are back: 2,799 in `workclass`, 2,809 in `occupation`, and 857 in `native-country`. Before moving on, check that the numbers make sense together, because the pattern of what's missing often tells you *why* it's missing.

**Ask:** `occupation` has exactly 10 more missing values than `workclass`. What would you check to explain the difference? Make a guess before you open the answer.

<details>
<summary>Answer</summary>

Compare *which rows* are missing in each column. First check whether every row missing `workclass` is also missing `occupation`. Then look at the rows that are missing only `occupation`. The next cells do both.

</details>

**Check 1: are they the same rows?** `.isna()` is `True` where a value is missing, and `&` keeps `True` only where both columns are `True`. So this counts the rows missing *both* `workclass` and `occupation`.

In [ ]:
both_missing = raw['workclass'].isna() & raw['occupation'].isna()
both_missing.sum()

That's 2,799, exactly the number of rows missing `workclass`. So every row missing `workclass` is missing `occupation` too, and the 10 extra rows are missing *only* `occupation`.

**Check 2: what do those 10 rows have in common?** `.notna()` is the opposite of `.isna()`: `True` where a value is present. So `raw['occupation'].isna() & raw['workclass'].notna()` picks out the rows where `occupation` is missing but `workclass` isn't. Then `.loc[..., 'workclass']` keeps just their `workclass` values, and `.value_counts()` counts them.

In [ ]:
only_occupation_missing = raw['occupation'].isna() & raw['workclass'].notna()
raw.loc[only_occupation_missing, 'workclass'].value_counts()

All 10 are `Never-worked`. Someone who has never worked has no occupation, so here, missing means "doesn't apply," not "unknown." (Similarly, `born_in_us` is blank in exactly the 857 rows where `native-country` is missing.)

This is worth doing in the Kaggle project too. When a value is missing, ask whether another column already tells you what it should be, or why it's absent.

### Problem 2: a True/False column with gaps

Pandas did recognize the 857 blank fields in `born_in_us` as missing. But a True/False column should load as `bool`, so check its type.

In [ ]:
print(raw['born_in_us'].dtype)
raw['born_in_us'].value_counts(dropna=False)

A column holding only `True` and `False` would load as `bool`. Even one blank field makes `pd.read_csv` load the whole column as `object`: a column of mixed Python values, here `True`, `False`, and `NaN`. Nothing looks wrong yet. The trouble starts when you convert the column, or hand it to scikit-learn.

**Ask:** What do you expect `.astype(bool)` to do with the 857 missing values?

<details>
<summary>Answer</summary>

It turns every one of them into `True`, with no error and no warning. In Python, `bool(x)` is `True` for any number other than 0, and `NaN` counts as a number other than 0.

</details>

In [ ]:
converted = raw['born_in_us'].astype(bool)
print(f"Missing before: {raw['born_in_us'].isna().sum()}, after .astype(bool): {converted.isna().sum()}")
print(f"True before:    {raw['born_in_us'].eq(True).sum():,}, after .astype(bool): {converted.sum():,}")

The 857 missing values silently became `True`. You might try converting to strings with `.astype(str)`, but this has the same kind of problem: it turns `NaN` into the text `'nan'`, which `.isna()` no longer counts as missing.

**Next, let's see what happens if we leave the column alone.** That fails too, but at least it fails loudly. In Sections 2 and 4 you'll learn the standard way to prepare a categorical column: fill in the missing values with the text `'missing'` (an imputer), then turn each category into its own 0/1 column (an encoder). The cell below runs `born_in_us` through those two steps. Notice the error this block outputs.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

# The pipeline uses a SimpleImputer to fill missing values with 'missing' and 
# then applies OneHotEncoder to encode categorical variables
born_in_us_pipeline = make_pipeline(SimpleImputer(
    strategy='constant', # Fill missing values with a constant value
    fill_value='missing' # Use 'missing' as the placeholder for missing values
    ),
      OneHotEncoder()) # Encode categorical variables as one-hot vectors)
try:
    born_in_us_pipeline.fit(raw[['born_in_us']]) # Just apply the pipeline to the 'born_in_us' column
except TypeError as error:
    print('TypeError:', error)

**The fix:** convert the column on purpose, keeping the missing values missing. Then it's an ordinary categorical column, and Sections 2 and 4 handle it like any other.

**Code worth reading.** `.map(...)` replaces each value using a dictionary. A value that isn't in the dictionary, like `NaN`, becomes `NaN`, so the missing values stay missing.

In [ ]:
raw['born_in_us'] = raw['born_in_us'].map({True: 'yes', False: 'no'})
raw['born_in_us'].value_counts(dropna=False) # dropna=False to include missing values in the count

**What the made-up column taught us.** `born_in_us` stands in for a True/False column with missing values:

- **How to spot it.** A column you expect to be `bool` loads as `object`. Check `.dtypes` right after `pd.read_csv`, then `value_counts(dropna=False)` to see what's mixed in.
- **Two conversions that fail silently.** `.astype(bool)` turned the 857 missing values into `True`, so the `True` count jumped from 43,832 to 44,689. `.astype(str)` turned them into the text `'nan'`, which `.isna()` no longer counts as missing. Neither raises an error, so you'd never know.
- **Leaving the column alone fails loudly.** The encoder raised a `TypeError` because the column mixed `True`/`False` with the imputer's text `'missing'`.
- **The fix.** Convert on purpose with `.map({True: 'yes', False: 'no'})`. It keeps the missing values missing, so the column becomes an ordinary categorical column. [Section 4](#section-4) finishes the job: it fills in the gaps with `'missing'` and one-hot encodes the column, so the pipeline that failed above runs properly.

**Check the repair.** With both fixes in place, the raw file's missing values should match OpenML's exactly (we'll omit the made up column, `born_in_us` in this comparison).

In [ ]:
pd.DataFrame({'raw file, fixed': raw.drop(columns='born_in_us' # drop the column we made up for the demo
                                          ).isna().sum(), # Count missing values 
              'OpenML': X.isna().sum()})

They match. `born_in_us` only summarizes `native-country`, which we already have, so we leave it behind and use `X` for the rest of the notebook.

**The habit to take away:** right after every `pd.read_csv`, check `.dtypes` and `.isna().sum()`, and ask two questions. Does every column have the type you expect? Is any missing value written as a code that pandas doesn't know? 

<a id="section-1"></a>

## 1 · Look first

Before any model, spend time examining the data. 
- What type is each column? 
- What's missing? 
- What looks odd? 
- How big is each class?

**Code worth reading.** `.info()` lists each column's type and how many values are present. `.isna().sum()` counts the missing values in each column.

In [ ]:
X.info()

In [ ]:
X.isna().sum()

Three columns have missing values, and all three are categorical. (In the original file, missing values were written as `?`. OpenML already turned them into proper missing values, `NaN`; Section 0 showed what happens when they're still written as `?`.)

**Code worth reading.** `.describe()` summarizes the numeric columns. Look at each column's minimum and maximum: do they make sense?

In [ ]:
X.describe().round(1)

In [ ]:
y.value_counts(normalize=True).round(3)      # the share of each class

**Ask:** Two things in these outputs deserve a closer look. What are they?

<details>
<summary>Answer</summary>

First, `capital-gain` has a maximum of 99,999, far above everything else (its 75th percentile is 0). Section 3 looks at that. Second, the classes are unbalanced: only about 24% of people earn above \$50K. That's why Section 5 compares accuracy with the majority-class floor.

</details>

<a id="section-2"></a>

## 2 · Missing values

There are two basic choices:

| Choice | When it's reasonable | The cost |
| --- | --- | --- |
| **Drop the rows** | Few rows are affected, or the *target* is missing (we can't learn from a row with no answer) | Throws away data. And the test set will have missing values too, and you still have to predict for those rows. |
| **Fill them in (impute)** | Most other cases | The filled-in values are guesses. |

**Imputation** fills in a missing value with a guess: the column's median for a number, the most common category for a category, or a new category that says "missing." That last choice is often best for categories, because *being missing* can itself carry information.

**Code worth reading.** `SimpleImputer(strategy='constant', fill_value='missing')` replaces each missing value with the text `'missing'`.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='constant', fill_value='missing')
filled = imputer.fit_transform(X[['occupation']]) # now, filled is a NumPy array with 'missing' replacing any NaN values
print(f"'missing' now appears {(filled == 'missing').sum()} times; NaN appears {pd.isna(filled).sum()} times")

**An imputer learns from data.** `SimpleImputer(strategy='median')` has to compute the median, and it computes it from whatever data you `fit` it on. If you fit it on *all the data* before cross-validating, every fold's "training" median has peeked at that fold's validation rows. This is called **data leakage**: information from the validation rows has leaked into training (think about why this is a problem). To avoid it, the imputer has to be fit inside each fold, on that fold's training rows only. A pipeline does that for you (Section 5).

**Ask:** Is `occupation` missing at random, or could missing occupations tell us something? Check the income rate for the missing rows.

<details>
<summary>Answer</summary>

Run `y[X['occupation'].isna()].mean()`: about 9% of the people with a missing occupation earn above \$50K, against 24% overall. Being missing carries information (many of these people aren't working), which is why keeping a separate `'missing'` category beats filling in the most common occupation.

</details>

In [ ]:
print(f"Income above $50K, occupation missing: {y[X['occupation'].isna()].mean():.1%}")
print(f"Income above $50K, everyone:           {y.mean():.1%}")

<a id="section-3"></a>

## 3 · Outliers: ask what the value means

An **outlier** is a value far from the rest. Plot first, then ask what the value *means* before you do anything. Most people have a capital gain of 0, so the plot shows only the nonzero values.

In [ ]:
from checkpoint_1_helpers import plot_capital_gain

plot_capital_gain(X['capital-gain']);

In [ ]:
top_coded = X['capital-gain'] == 99_999
print(f'{top_coded.sum()} people at exactly 99,999; the next largest value is '
      f"{X.loc[~top_coded, 'capital-gain'].max():,}")
print(f'Income above $50K among them: {y[top_coded].mean():.0%}')

Nobody's capital gain lands on exactly \$99,999 by chance, 244 times, with nothing between \$41,310 and \$99,999. It's a code: most likely the census recorded every gain at or above some cap as 99,999 (this is called **top-coding**). These are real people with very large gains, and every one of them earns above \$50K.

What to do, in order of preference:

- **Don't drop by default.** These rows hold real information, and the test data will have them too.
- **Know your model.** Trees barely care about extreme values: a split only asks which side of $s$ a value is on. Linear models can be pulled hard by them; standardizing doesn't fix that.
- **If it matters**, cap the column: replace every value above a limit with the limit itself, using `.clip(upper=...)`. Capping at 41,310, the largest real value, keeps these people at the top of the column but stops them from pulling so hard on a linear model. Or add a 0/1 column for "is top-coded." Compare the choices with cross-validation (you'll see how before Checkpoint 2).
- **Watch for leakage.** A fixed limit, like 41,310 read off the plot, is fine to apply before cross-validation. A limit *computed* from the data, like "the 99th percentile," is learned from the data, just like an imputer's median, so it has to be computed inside each fold (Section 2).

For simplicity we'll leave the column as is, but you might want to try other approaches if you see a similar situation in the Kaggle data.

**Ask:** A classmate deletes the 244 rows at 99,999 "because they're outliers." What goes wrong?

<details>
<summary>Answer</summary>

Those rows carry a strong signal (all of them earn above \$50K), so the model loses it. And the test data will still contain people at 99,999, which the model will now have never seen. Deleting is only right when a value is a genuine error, and even then, the test set may have the same error.

</details>

<a id="section-4"></a>

## 4 · Categorical features: one-hot encoding

Models compute with numbers. Why not number the categories, say `Private` = 0, `Self-emp-not-inc` = 1, `Local-gov` = 2? For a linear model, that says the categories are in order and equally spaced: that `Local-gov` is "twice" `Self-emp-not-inc`, and that one coefficient covers the whole list. Nothing about job types justifies that.

**One-hot encoding** makes a separate column per category containing only 0s and 1s instead. Each row has a 1 in its own category's column and 0 everywhere else, so each category (now a one-hot encoded column) gets its own model coefficient.

**Code worth reading.** `handle_unknown='ignore'` matters on real data: if a category shows up in new data that wasn't in the training data, its row gets all zeros instead of an error. `sparse_output=False` returns an ordinary table of numbers rather than a memory efficient format.

<details>
<summary>Why sparse output?</summary>

By default, `OneHotEncoder` returns a **sparse matrix** instead: a compressed format that stores only the nonzero entries, which saves memory because one-hot columns are mostly zeros. We ask for the ordinary table because it's easier to look at: `pd.DataFrame(...)` can't wrap a sparse matrix directly. Many models accept sparse input, but some, like scikit-learn's fast boosting model `HistGradientBoostingClassifier`, stop with a `TypeError`. If you try one of those models in the Kaggle project and see that error, `sparse_output=False` is the fix.

</details>

In [ ]:
from sklearn.preprocessing import OneHotEncoder

small = pd.DataFrame({'workclass': ['Private', 'Local-gov', 'Private', 'State-gov']})
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(small)
pd.DataFrame(encoder.transform(small), columns=encoder.get_feature_names_out()).astype(int)

In [ ]:
new_data = pd.DataFrame({'workclass': ['Federal-gov']})     # not seen during fit
encoder.transform(new_data)                                 # all zeros, no error

A two-valued category needs only one column (`OneHotEncoder(drop='if_binary')`). A column with many categories, like `native-country` (41 countries), makes many columns; that's fine for most models, but worth knowing.

**Back to `born_in_us`.** In Section 0, the two steps that fill in missing categories with `'missing'` and then one-hot encode them failed, because the column mixed `True`/`False` with text. After the `.map` fix, every value is `'yes'`, `'no'`, or missing, so the same two steps work. `raw` is still the data from Section 0, with `born_in_us` converted.

**Code worth reading.**

- **The important part** is the pipeline and `fit_transform`: it fills in the gaps with `'missing'`, then one-hot encodes the result. `raw[['born_in_us']]` has double brackets so the pipeline receives a one-column table, not a single column.
- **The rest is just for readability (you can ignore this).** In a real project, the pipeline's output would go straight to a model. Here we want to see it, so `pd.DataFrame(...)` puts the numbers in a table; `get_feature_names_out()` labels its columns (`born_in_us_pipeline[-1]` is the pipeline's last step, the encoder, and `x0` is just scikit-learn's name for the first input column); `.astype(int)` shows `1` and `0` instead of `1.0` and `0.0`; and the last line lists each distinct value of `born_in_us` next to the 0/1 columns it becomes.

In [ ]:
born_in_us_pipeline = make_pipeline(SimpleImputer(strategy='constant', fill_value='missing'),
                                    OneHotEncoder(sparse_output=False))
encoded = pd.DataFrame(born_in_us_pipeline.fit_transform(raw[['born_in_us']]),
                       columns=born_in_us_pipeline[-1].get_feature_names_out()).astype(int) # <- just to make a pretty table
pd.concat([raw['born_in_us'], encoded], axis=1).drop_duplicates()

Three columns, one per category: `yes`, `no`, and `missing`. The rows with a gap get their own 1, so a model can learn whether being missing matters (Section 2). Section 5 does these steps for every categorical column at once.

<a id="section-5"></a>

## 5 · One pipeline

### A tiny pipeline first

A **pipeline** chains several steps into one object that runs them in order. The steps are usually preprocessing (imputing, scaling, encoding) followed by a model. Here's a tiny one before the real one. The data: eight made-up people, with their weekly hours (two are missing) and whether they earn above \$50K. `hours` has one row per person and one column.

In [ ]:
hours = np.array([[20], [35], [np.nan], [40], [45], [np.nan], [55], [60]])
above_50k = np.array([0, 0, 0, 0, 1, 1, 1, 1])          # 1 = earns above $50K
new_hours = np.array([[np.nan], [58]])                   # two new people; one has missing hours

**Without a pipeline**, you manage the steps yourself: fit the imputer on the training rows, use it to fill in the gaps, then fit the model on the result. And every time you predict for new people, you have to remember to send them through the same fitted imputer first.

In [ ]:
from sklearn.linear_model import LogisticRegression

imputer = SimpleImputer(strategy='median').fit(hours)
model = LogisticRegression().fit(imputer.transform(hours), # the transformed data are used for training
                                 above_50k)

try:
    model.predict(new_hours)                              # we forgot to impute new_hours first
except ValueError as error:
    print('ValueError:', str(error).split('\n')[0])
print('Predictions:', model.predict(imputer.transform(new_hours)))

Forgetting that step is an easy mistake, and a bigger project has more steps to forget. **With a pipeline**, you list the steps once.

In [ ]:
tiny_pipeline = make_pipeline(SimpleImputer(strategy='median'), LogisticRegression())
tiny_pipeline.fit(hours, above_50k)                       # fits the imputer, then the model

print('Learned median:', tiny_pipeline[0].statistics_)   # [0] is the first step, the imputer
print('Predictions:   ', tiny_pipeline.predict(new_hours))

- `make_pipeline(...)` only builds the pipeline. `.fit(X, y)` is what fits it: the imputer learns the median from the training rows, fills in the gaps, and the model is fit on the result.
- `.predict(...)` then runs the same steps for you, using what was learned from the training rows. There's no step to forget.
- That's also what keeps cross-validation honest. `cross_val_score` fits the whole pipeline again on each fold's training rows, so every learned value (here, the median) comes from that fold's training rows only. That's the leak Section 2 warned about, fixed automatically.

### The real pipeline

Now put every step together. Numeric columns get imputed (with the median) and standardized; categorical columns get imputed (with `'missing'`) and one-hot encoded. Then a model.

**Code worth reading.**

- `make_pipeline(A, B)` chains steps, as in the tiny pipeline above. Here, `cross_val_score` does the fitting, once per fold.
- `make_column_transformer` sends each group of columns through its own steps and puts the results side by side.
- The numeric columns have no missing values in this data, but the imputer stays: new data might have some.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numeric_columns = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_columns = ['workclass', 'marital-status', 'occupation', 'relationship', 'native-country']

numeric_pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical_pipeline = make_pipeline(SimpleImputer(strategy='constant', fill_value='missing'),
                                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
preprocess = make_column_transformer((numeric_pipeline, numeric_columns),
                                     (categorical_pipeline, categorical_columns),
                                     verbose_feature_names_out=False) # Prevents adding prefixes to column names

logistic_pipeline = make_pipeline(preprocess, LogisticRegression(max_iter=1000))

**Code worth reading.** Cross-validate the whole pipeline. In each of the 5 folds, `cross_val_score` fits the imputers, the scaler, the encoder, and the model on that fold's training rows only, then scores the validation rows. `DummyClassifier` always predicts the most common class: it's the **majority-class floor**, the accuracy you get without looking at any features. (`max_iter=1000` gives logistic regression enough steps to finish.)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=5, shuffle=True, # It only shuffles once before creating the folds
               random_state=0)
floor = cross_val_score(DummyClassifier(), X, y, cv=folds)
logistic_scores = cross_val_score(logistic_pipeline, X, y, cv=folds)
print(f'Majority-class floor:  {floor.mean():.1%}')
print(f'Logistic regression:   {logistic_scores.mean():.1%} ± {logistic_scores.std():.1%}')

<a id="section-6"></a>

## 6 · Summary: a checklist for a new data set

0. **Load carefully:** right after `pd.read_csv`, check `.dtypes` and `.isna().sum()`. Pass any missing-value codes with `na_values`, and convert True/False columns with gaps on purpose (for example with `.map`), never with `.astype(bool)`.
1. **Look first:** `.info()`, `.isna().sum()`, `.describe()`, and the class shares. Look for codes that mean "missing" and values that can't be right.
2. **Missing values:** drop rows only when the target is missing (or very few rows are affected); otherwise impute. A `'missing'` category is often best for categorical columns.
3. **Outliers:** plot, then ask what the value means. Don't drop by default. Trees barely care; linear models can.
4. **Categories:** one-hot encode, with `handle_unknown='ignore'`.
5. **One pipeline:** imputers, scaler (not needed for trees), encoder, and the model, all inside one pipeline, cross-validated with the same folds for every model.

# Practice Questions

These questions cover reading raw files, missing values, outliers, and encoding. Try each one on paper before you open its answer.

### Question 1: Reading a raw file

A raw CSV file has a column `is_member` holding `True`, `False`, and some blank fields. (See [Section 0](#section-0).)

(a) What type does `pd.read_csv` give the column?

(b) A classmate runs `df['is_member'].astype(bool)`. What happens to the blank fields?

(c) In the same file, missing ages are written as `-1`. How do you make `pd.read_csv` treat them as missing?

<details>
<summary>Answer</summary>

(a) **`object`**, a column of mixed Python values (`True`, `False`, and `NaN`), not `bool`.

(b) **They all become `True`**, silently. `bool(NaN)` is `True`, so the missing values disappear and inflate the `True` count. Convert on purpose instead, for example with `.map({True: 'yes', False: 'no'})`, which keeps them missing.

(c) **`pd.read_csv(path, na_values=-1)`**, or a list such as `na_values=['?', -1]` if the file uses several codes. Check `.isna().sum()` afterward.

</details>

### Question 2: Drop or impute?

For each situation, would you drop the rows or impute? (See [Section 2](#section-2).)

(a) 59 rows have no value for the target.

(b) 6% of rows have no value for one categorical feature, and the test set has missing values in the same column.

<details>
<summary>Answer</summary>

(a) **Drop.** A row with no target has nothing to learn from.

(b) **Impute**, for example with a `'missing'` category. Dropping throws away 6% of the data, and you'd still have to predict for the test rows with missing values.

</details>

### Question 3: Where to fit the imputer

A classmate fits `SimpleImputer(strategy='median')` on the whole training set, then runs 5-fold cross-validation on the imputed data. What's wrong, and how do you fix it? (See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

**The medians were computed using every fold's validation rows**, so the validation rows weren't truly new. Put the imputer in a pipeline with the model, and cross-validate the pipeline, so the median is recomputed from each fold's training rows only.

</details>

### Question 4: An odd value

244 people have a capital gain of exactly 99,999, and the next largest value is 41,310. What's the most likely explanation, and should you delete those rows? (See [Section 3](#section-3).)

<details>
<summary>Answer</summary>

**It's a code (top-coding): gains at or above a cap were recorded as 99,999.** Don't delete the rows: they're real people, they carry a strong signal (all earn above \$50K), and the test data will contain them too.

</details>

### Question 5: Numbering categories

A column `color` has values red, green, and blue. A classmate codes them 0, 1, 2 and fits a linear regression.

(a) What does that coding assume?

(b) How many 0/1 columns does one-hot encoding make for this column (without dropping any)? (See [Section 4](#section-4).)

<details>
<summary>Answer</summary>

(a) **That the colors are in order and equally spaced**, so one coefficient covers them all and blue's effect is exactly twice green's.

(b) **3**, one per color.

</details>